# Section 7 — Adaptive RK23 and Comparison of Methods

[← Classical RK4](06_rk4.ipynb) · [Module 3](README.md)

Fixed-step methods use the same $h$ everywhere. Adaptive methods estimate local
error and change the step size while the computation is running.

This notebook implements the adaptive RK2(3) construction used in Brin and then
compares the principal methods from Module 3 on one IVP.

### Learning goals

- understand an embedded RK2(3) pair;
- estimate local error from two formulas sharing the same stages;
- accept or reject a trial step;
- adjust the step size;
- compare Euler, Taylor 2/3/4, RK2, RK3, RK4, and adaptive RK23.


In [1]:
import math

import matplotlib.pyplot as plt
import numpy as np


## 1. Common test problem

Use

$$
y'=t e^{3t}-2y,
\qquad
y(0)=0,
\qquad
0\le t\le1.
$$

The exact solution is

$$
y(t)
=
e^{3t}
\left(
\frac{t}{5}
-\frac1{25}
\right)
+
\frac1{25}e^{-2t}.
$$


In [2]:
def f(t, y):
    return t * math.exp(3.0 * t) - 2.0 * y


def exact(t):
    return (
        np.exp(3.0 * t)
        * (t / 5.0 - 1.0 / 25.0)
        + np.exp(-2.0 * t) / 25.0
    )


def y2(t, y):
    return (
        math.exp(3.0 * t) * (1.0 + t)
        + 4.0 * y
    )


def y3(t, y):
    return (
        math.exp(3.0 * t) * (4.0 + 7.0 * t)
        - 8.0 * y
    )


def y4(t, y):
    return (
        math.exp(3.0 * t) * (19.0 + 13.0 * t)
        + 16.0 * y
    )


## 2. Fixed-step methods

The implementations below all use the same function $f(t,y)$ and the same
interval so that the final comparison is meaningful.


In [3]:
def solve_fixed(step_function, h):
    """Apply a one-step method from t=0 to t=1."""
    n_steps = round(1.0 / h)
    t = 0.0
    y = 0.0

    for _ in range(n_steps):
        y = step_function(t, y, h)
        t += h

    return y


def euler_step(t, y, h):
    return y + h * f(t, y)


def taylor2_step(t, y, h):
    return (
        y
        + h * f(t, y)
        + h**2 * y2(t, y) / 2.0
    )


def taylor3_step(t, y, h):
    return (
        y
        + h * f(t, y)
        + h**2 * y2(t, y) / 2.0
        + h**3 * y3(t, y) / 6.0
    )


def taylor4_step(t, y, h):
    return (
        y
        + h * f(t, y)
        + h**2 * y2(t, y) / 2.0
        + h**3 * y3(t, y) / 6.0
        + h**4 * y4(t, y) / 24.0
    )


def improved_euler_step(t, y, h):
    k1 = f(t, y)
    k2 = f(
        t + h,
        y + h * k1,
    )
    return y + h * (k1 + k2) / 2.0


def midpoint_step(t, y, h):
    k1 = f(t, y)
    k2 = f(
        t + h / 2.0,
        y + h * k1 / 2.0,
    )
    return y + h * k2


def heun_rk3_step(t, y, h):
    k1 = f(t, y)
    k2 = f(
        t + h / 3.0,
        y + h * k1 / 3.0,
    )
    k3 = f(
        t + 2.0 * h / 3.0,
        y + 2.0 * h * k2 / 3.0,
    )
    return y + h * (k1 + 3.0 * k3) / 4.0


def rk4_step(t, y, h):
    k1 = f(t, y)
    k2 = f(
        t + h / 2.0,
        y + h * k1 / 2.0,
    )
    k3 = f(
        t + h / 2.0,
        y + h * k2 / 2.0,
    )
    k4 = f(
        t + h,
        y + h * k3,
    )
    return y + h * (
        k1
        + 2.0 * k2
        + 2.0 * k3
        + k4
    ) / 6.0


## 3. Brin-style adaptive RK2(3)

The three stages are

$$
k_1=f(t,y),
$$

$$
k_2=
f\left(
t+\frac h3,
y+\frac h3k_1
\right),
$$

$$
k_3=
f\left(
t+\frac{2h}{3},
y+\frac{2h}{3}k_2
\right).
$$

The third-order approximation is

$$
w_3
=
y+\frac h4(k_1+3k_3),
$$

while the embedded second-order approximation is

$$
w_2
=
y+\frac h2(k_2+k_3).
$$

Their difference gives the error estimate

$$
R
=
|w_3-w_2|
=
\left|
\frac h4
(k_1-2k_2+k_3)
\right|.
$$

A trial step is accepted when $R$ is below the requested tolerance. Otherwise
the step is rejected and repeated with a smaller $h$.


In [4]:
def rk23_adaptive(
    function,
    t0,
    y0,
    t_final,
    h_initial=0.2,
    tolerance=1.0e-5,
    safety=0.9,
):
    """Adaptive RK2(3) using a shared three-stage Heun pair."""
    t = t0
    y = y0
    h = h_initial

    t_values = [t]
    y_values = [y]
    h_values = []
    error_estimates = []

    function_evaluations = 0

    while t < t_final:
        h = min(h, t_final - t)

        k1 = function(t, y)
        k2 = function(
            t + h / 3.0,
            y + h * k1 / 3.0,
        )
        k3 = function(
            t + 2.0 * h / 3.0,
            y + 2.0 * h * k2 / 3.0,
        )
        function_evaluations += 3

        y_order3 = y + h * (
            k1 + 3.0 * k3
        ) / 4.0

        y_order2 = y + h * (
            k2 + k3
        ) / 2.0

        error_estimate = abs(
            y_order3 - y_order2
        )

        if error_estimate <= tolerance:
            t += h
            y = y_order3

            t_values.append(t)
            y_values.append(y)
            h_values.append(h)
            error_estimates.append(
                error_estimate
            )

        if error_estimate == 0.0:
            factor = 2.0
        else:
            factor = safety * (
                tolerance / error_estimate
            ) ** (1.0 / 3.0)
            factor = min(
                5.0,
                max(0.2, factor),
            )

        h *= factor

    return {
        "t": np.array(t_values),
        "y": np.array(y_values),
        "h": np.array(h_values),
        "error": np.array(error_estimates),
        "function_evaluations": function_evaluations,
    }


In [5]:
rk23_result = rk23_adaptive(
    f,
    0.0,
    0.0,
    1.0,
    h_initial=0.2,
    tolerance=1.0e-5,
)

rk23_error = abs(
    exact(1.0)
    - rk23_result["y"][-1]
)

print(
    "accepted steps       = "
    f"{len(rk23_result['h'])}"
)
print(
    "function evaluations = "
    f"{rk23_result['function_evaluations']}"
)
print(
    "RK23 approximation   = "
    f"{rk23_result['y'][-1]:.10f}"
)
print(
    "actual final error   = "
    f"{rk23_error:.3e}"
)


accepted steps       = 40
function evaluations = 123
RK23 approximation   = 3.2190924295
actual final error   = 6.890e-06


In [6]:
plt.figure(figsize=(9, 4.5))
plt.step(
    rk23_result["t"][1:],
    rk23_result["h"],
    where="post",
)
plt.xlabel("t")
plt.ylabel("accepted step size h")
plt.title("Adaptive RK23 Step Sizes")
plt.grid(alpha=0.25)
plt.show()


## 4. Compare all methods

Use fixed step size

$$
h=0.1
$$

for the fixed-step methods. Adaptive RK23 chooses its own step sizes.

The table is not meant to declare one universal winner. It illustrates how
higher-order methods usually achieve smaller error for the same fixed $h$, while
adaptive methods spend work where their error estimator says it is needed.


In [7]:
h = 0.1

fixed_methods = {
    "Euler": euler_step,
    "Taylor 2": taylor2_step,
    "Taylor 3": taylor3_step,
    "Taylor 4": taylor4_step,
    "Improved Euler": improved_euler_step,
    "Midpoint RK2": midpoint_step,
    "Heun RK3": heun_rk3_step,
    "RK4": rk4_step,
}

comparison = []

for name, step_function in fixed_methods.items():
    approximation = solve_fixed(
        step_function,
        h,
    )
    error = abs(
        exact(1.0)
        - approximation
    )
    comparison.append(
        (
            name,
            approximation,
            error,
        )
    )

comparison.append(
    (
        "Adaptive RK23",
        rk23_result["y"][-1],
        rk23_error,
    )
)

print(
    "method              approximation       "
    "absolute error"
)

for name, approximation, error in comparison:
    print(
        f"{name:18s} "
        f"{approximation:14.9f}   "
        f"{error:12.4e}"
    )


method              approximation       absolute error
Euler                 2.760901468     4.5820e-01
Taylor 2              3.161442560     5.7657e-02
Taylor 3              3.213955856     5.1435e-03
Taylor 4              3.218741477     3.5784e-04
Improved Euler        3.297890508     7.8791e-02
Midpoint RK2          3.225416283     6.3170e-03
Heun RK3              3.218170660     9.2866e-04
RK4                   3.219283395     1.8408e-04
Adaptive RK23         3.219092429     6.8896e-06


In [8]:
names = [
    item[0]
    for item in comparison
]
errors = np.array(
    [
        item[2]
        for item in comparison
    ]
)

plt.figure(figsize=(10, 5))
plt.semilogy(
    names,
    errors,
    "o",
)
plt.xticks(
    rotation=45,
    ha="right",
)
plt.ylabel("absolute error at t=1")
plt.title("Comparison of Module 3 Methods")
plt.grid(
    axis="y",
    alpha=0.25,
)
plt.tight_layout()
plt.show()


## Main comparison ideas

- Euler is first order.
- Taylor 2 and RK2 methods are second order.
- Taylor 3 and Heun RK3 are third order.
- Taylor 4 and classical RK4 are fourth order.
- RK methods obtain higher order without explicit higher derivatives of $y$.
- Adaptive RK23 changes $h$ in response to an estimated local error.

### References

- Leon Q. Brin, *Tea Time Numerical Analysis*, Chapter 6.
- Richard L. Burden and J. Douglas Faires, *Numerical Analysis*.
- MAD 4401 Runge-Kutta lectures and Exam 3 review.

[← Classical RK4](06_rk4.ipynb) · [↑ Module 3](README.md)
